In [10]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_PATH = Path("C:/Users/Архиповы/source/repos/modem")
sys.path.insert(0, str(PROJECT_PATH / "src"))

from modem.blocks import MessageSource, StringToBits, PacketBuileder, PCM, BPSK, DPSK, HammingDecoder, HammongEncoder
from modem.blocks import BFSK, Channel, BFSKDemodulator, PacketDecoder, Demodulator, BitsToString, DPSKDemodulator, BFSKCoherentDemodulator
from modem.pipeline import run

MAX_DISPLAY_SAMPLES = 2000
MAX_DISPLAY_BITS = 30

def time_graph(signal, fs, name, ax=None, is_step=False):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(9, 2.5))
    n_show = min(len(signal), MAX_DISPLAY_SAMPLES)
    t = np.arange(n_show) / fs
    if is_step:
        ax.step(t, signal[:n_show], where='post', color='blue', linewidth=1)
    else:
        ax.plot(t, signal[:n_show], color='blue', linewidth=1)
    ax.grid(True, alpha=0.3)
    ax.set_title(name, fontsize=10)
    return ax

def spectrum_graph(signal, fs, name, ax=None, max_samples=2000):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(9, 2.5))
    n = min(len(signal), max_samples)
    signal_short = signal[:n]
    fft_vals = np.fft.fft(signal_short)
    amp = np.abs(fft_vals) / n
    freq = np.fft.fftfreq(n, 1/fs)
    ax.plot(freq[:n//2], amp[:n//2], color='red', linewidth=1)
    ax.grid(True, alpha=0.3)
    ax.set_title(name, fontsize=10)
    return ax

def calculate_ber(original_bits, recovered_bits):
    if len(original_bits) == 0 or len(recovered_bits) == 0:
        return 1.0
    min_len = min(len(original_bits), len(recovered_bits))
    if min_len == 0:
        return 1.0
    errors = np.sum(original_bits[:min_len] != recovered_bits[:min_len])
    return errors / min_len

mod_type = widgets.RadioButtons(
    options=['BPSK', 'BFSK', 'DPSK'], 
    value='BPSK', 
    description='Модуляция:'
)
demod_type = widgets.RadioButtons(
    options=['Когерентный', 'Некогерентный'],
    value='Когерентный',
    description='Тип BFSK демодуляции:'
)

fc_slider = widgets.IntSlider(value=300, min=50, max=5000, step=50, description='Fc (Гц):')
Rb_slider = widgets.IntSlider(value=100, min=10, max=500, step=10, description='R_b (бит/с):')
h_slider = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='h (BFSK):')
k_slider = widgets.FloatSlider(value=0.95, min=0.1, max=1.0, step=0.01, description='Затухание:')
snr_slider = widgets.FloatSlider(value=30, min=-20, max=50, step=1, description='SNR (дБ):')
noise_type = widgets.Dropdown(options=['gaussian', 'laplace', 'none'], value='gaussian', description='Тип шума:')
phase_noise_slider = widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.01, description='Фазовый шум:')
use_doppler = widgets.Checkbox(value=False, description='Эффект Доплера')
velocity_slider = widgets.FloatSlider(value=30.0, min=0.0, max=100.0, step=1.0, description='Скорость (м/с):')
angle_slider = widgets.FloatSlider(value=0.0, min=0.0, max=3.14, step=0.1, description='Угол (рад):')
use_hamming = widgets.Checkbox(value=True, description='Код Хэмминга')
fs_choice = widgets.RadioButtons(
    options=[('Хорошая (20·Fc)', 'good'), ('Плохая (2.1·Fc)', 'bad')],
    value='good',
    description='Дискретизация:'
)

source_type = widgets.RadioButtons(
    options=[('Ввести текст', 'text'), ('Загрузить из файла', 'file')], 
    value='text', 
    description='Источник:'
)
text_input = widgets.Textarea(value='Hello DSP!', description='Текст:', layout=widgets.Layout(width='100%', height='80px'))
file_input = widgets.Text(value='message.txt', description='Файл:', disabled=True)

def update_source(*args):
    file_input.disabled = (source_type.value != 'file')
source_type.observe(update_source, 'value')

run_button = widgets.Button(description='Старт', button_style='success', layout=widgets.Layout(width='200px'))

left_panel = widgets.VBox(
    [
        widgets.HTML("<h3>Настройки параметров передачи</h3>"),
        mod_type,
        demod_type,
        fs_choice,
        fc_slider,
        Rb_slider,
        h_slider,
        widgets.HTML("<hr>"),
        k_slider,
        snr_slider,
        noise_type,
        phase_noise_slider,
        use_doppler,
        velocity_slider,
        angle_slider,
        use_hamming,
        widgets.HTML("<hr>"),
        source_type,
        text_input,
        file_input,
        widgets.HTML("<hr>"),
        run_button
    ],
    layout=widgets.Layout(
        width='320px', min_width='320px', max_width='320px',
        border='1px solid gray', padding='10px'
    )
)

right_panel = widgets.Output(
    layout=widgets.Layout(flex='1 1 auto', overflow='auto', padding='10px')
)

main_ui = widgets.HBox(
    [left_panel, right_panel],
    layout=widgets.Layout(width='100%', align_items='flex-start')
)

display(main_ui)

def run_simulation(b):
    with right_panel:
        clear_output(wait=True)
        
        mod_type_val = mod_type.value
        fc = fc_slider.value
        R_b = Rb_slider.value
        k = k_slider.value
        snr_db = snr_slider.value
        noise_type_val = noise_type.value
        phase_noise_std = phase_noise_slider.value
        use_doppler_val = use_doppler.value
        velocity = velocity_slider.value
        angle = angle_slider.value
        use_hamming_val = use_hamming.value
        h = h_slider.value
        fs_val = fs_choice.value
        
        if fs_val == 'good':
            samp_rate = fc * 20
        else:
            samp_rate = int(fc * 2.1)

        samp_per_bit = int(samp_rate / R_b)
        
        f0 = fc - R_b * h
        f1 = fc + R_b * h
        
        if source_type.value == 'text':
            text = text_input.value
            filepath = None
        else:
            filepath = file_input.value
            text = None
            if not Path(filepath).exists():
                print(f"Файл {filepath} не найден")
                return
        
        ctx = {}
        capture = {}
        
        blocks = [
            MessageSource(filepath=filepath, default_text=text or "Hello", name="MS"),
            StringToBits(encoding='utf-8', name="StB"),
        ]
        
        if use_hamming_val:
            blocks.append(HammongEncoder(name="HE"))
        
        blocks.append(PacketBuileder(preamble="1010101010101010", sfd="10101011", add_crc=True, name="packet"))
        blocks.append(PCM(name="PCM"))
        
        if mod_type_val == "BPSK":
            blocks.append(BPSK(freq_carrier=fc, R_b=R_b, samp_rate=samp_rate, name="bpsk"))
            blocks.append(Channel(
                k=k, snr_db=snr_db, noise_type=noise_type_val,
                phase_noise_std=phase_noise_std,
                use_doppler=use_doppler_val,
                velocity=velocity, angle=angle,
                fc=fc, samp_rate=samp_rate,
                name="channel"
            ))
            blocks.append(Demodulator(freq_carrier=fc, samp_rate=samp_rate, R_b=R_b, lpf_cutoff_multiplier=1.5, name="demod"))
            
        elif mod_type_val == "DPSK":
            blocks.append(DPSK(freq_carrier=fc, R_b=R_b, samp_rate=samp_rate, name="dpsk"))
            blocks.append(Channel(
                k=k, snr_db=snr_db, noise_type=noise_type_val,
                phase_noise_std=phase_noise_std,
                use_doppler=use_doppler_val,
                velocity=velocity, angle=angle,
                fc=fc, samp_rate=samp_rate,
                name="channel"
            ))
            blocks.append(DPSKDemodulator(freq_carrier=fc, samp_rate=samp_rate, R_b=R_b, name="dpsk_demod"))
            
        else:  # BFSK
            blocks.append(BFSK(freq_carrier=fc, R_b=R_b, samp_rate=samp_rate, h=h, name="BFSK"))
            blocks.append(Channel(
                k=k, snr_db=snr_db, noise_type=noise_type_val,
                phase_noise_std=phase_noise_std,
                use_doppler=use_doppler_val,
                velocity=velocity, angle=angle,
                fc=fc, samp_rate=samp_rate,
                name="channel"
            ))
            blocks.append(BFSKDemodulator(samp_rate=samp_rate, R_b=R_b, f0=f0, f1=f1, name="BFSKDM"))
            '''if demod_type.value == "Когерентный":
                blocks.append(BFSKCoherentDemodulator(samp_rate=samp_rate, R_b=R_b, f0=f0, f1=f1, name="BFSKDM"))
            else:
                blocks.append(BFSKDemodulator(samp_rate=samp_rate, R_b=R_b, f0=f0, f1=f1, name="BFSKDM"))'''
        
        blocks.append(PacketDecoder(preamble="1010101010101010", sfd="10101011", name="PackDec"))
        
        if use_hamming_val:
            blocks.append(HammingDecoder(name="HammDecoder"))
        
        blocks.append(BitsToString(encoding='utf-8', name="BSt"))
        
        try:
            result = run(blocks, x=None, ctx=ctx, capture=capture)
        except Exception as e:
            print(f"Ошибка: {e}")
            return
        
        original_text = capture.get("MS", "???")
        decoded_text = capture.get("BSt", "???")
        original_bits = np.array(capture.get("StB", []))
        nrz_signal = np.array(capture.get("PCM", []))
        channel_out = np.array(capture.get("channel", []))
        
        if mod_type_val == "BPSK":
            mod_signal = np.array(capture.get("bpsk", []))
            before_lpf = np.array(capture.get("before_LPF", []))
            after_lpf = np.array(capture.get("after_LPF", []))
            envelope0 = None
            envelope1 = None
        elif mod_type_val == "DPSK":
            mod_signal = np.array(capture.get("dpsk", []))
            before_lpf = np.array([])
            after_lpf = np.array([])
            envelope0 = None
            envelope1 = None
        else:
            mod_signal = np.array(capture.get("BFSK", []))
            envelope0 = np.array(capture.get("bfsk_envelope0", []))
            envelope1 = np.array(capture.get("bfsk_envelope1", []))
            before_lpf = np.array([])
            after_lpf = np.array([])
        
        if use_hamming_val:
            recovered_bits = np.array(ctx.get("after_hamming", []))
        else:
            recovered_bits = np.array(ctx.get("decoded_bits", []))
        
        ber = calculate_ber(original_bits, recovered_bits)
        
        print(f"Отправлено: {original_text}")
        print(f"Получено: {decoded_text}")
        print(f"BER: {ber:.8f}")
        
        if ber == 0 and original_text == decoded_text:
            print("Сообщение передано без ошибок")
        else:
            print("Ошибки при передаче")
        
        # Графики
        if len(original_bits) > 0:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2.5))
            time_graph(original_bits, samp_rate, "Исходные биты", ax1, is_step=True)
            spectrum_graph(original_bits, samp_rate, "Спектр исходных битов", ax2)
            plt.tight_layout()
            plt.show()
        
        if len(nrz_signal) > 0:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2.5))
            time_graph(nrz_signal, samp_rate, "NRZ сигнал", ax1, is_step=True)
            spectrum_graph(nrz_signal, samp_rate, "Спектр NRZ", ax2)
            plt.tight_layout()
            plt.show()
        
        if len(mod_signal) > 0:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2.5))
            time_graph(mod_signal, samp_rate, f"{mod_type_val} сигнал", ax1)
            spectrum_graph(mod_signal, samp_rate, f"Спектр {mod_type_val}", ax2)
            plt.tight_layout()
            plt.show()
        
        if len(channel_out) > 0:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2.5))
            time_graph(channel_out, samp_rate, "Сигнал после канала", ax1)
            spectrum_graph(channel_out, samp_rate, "Спектр после канала", ax2)
            plt.tight_layout()
            plt.show()
        
        if mod_type_val == "BPSK" and len(before_lpf) > 0 and len(after_lpf) > 0:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2.5))
            time_graph(before_lpf, samp_rate, "Сигнал до ФНЧ", ax1)
            time_graph(after_lpf, samp_rate, "Сигнал после ФНЧ", ax2)
            plt.tight_layout()
            plt.show()
        elif mod_type_val == "BFSK" and len(envelope0) > 0 and len(envelope1) > 0:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2.5))
            n_show = min(len(envelope0), 1000)
            t = np.arange(n_show) / samp_rate
            ax1.plot(t, envelope0[:n_show], label='f0', color='blue', linewidth=1)
            ax1.plot(t, envelope1[:n_show], label='f1', color='red', linewidth=1)
            ax1.grid(True, alpha=0.3)
            ax1.legend(fontsize=8)
            ax1.set_title("Огибающие BFSK", fontsize=10)
            spectrum_graph(envelope0, samp_rate, "Спектр огибающей f0", ax2)
            plt.tight_layout()
            plt.show()
        
        if len(recovered_bits) > 0:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 2.5))
            n_show = min(len(recovered_bits), MAX_DISPLAY_BITS)
            t_bits = np.arange(n_show) * samp_per_bit / samp_rate
            ax1.step(t_bits, recovered_bits[:n_show], where='post', color='blue', linewidth=1)
            ax1.grid(True, alpha=0.3)
            ax1.set_title("Демодулированные биты", fontsize=10)
            ax1.set_ylim(-0.2, 1.2)
            spectrum_graph(recovered_bits, samp_rate, "Спектр демодулированных битов", ax2)
            plt.tight_layout()
            plt.show()

run_button.on_click(run_simulation) 